In [8]:
# Cell 1: Import modules
import torch
import torch.nn as nn
import pickle
import numpy as np

# Cell 2: Define the model structure (must match the trained model exactly)
class DeepSynergyModel(nn.Module):
    def __init__(self, input_size):
        super(DeepSynergyModel, self).__init__()
        self.model = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(input_size, 8192),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(8192, 4096),
            nn.ReLU(),
            nn.Linear(4096, 1)  # Output: predicted synergy score
        )

    def forward(self, x):
        return self.model(x)

# Cell 3: Load model
input_size = 12758  # verified from X.p
model = DeepSynergyModel(input_size=input_size)
model.load_state_dict(torch.load("deepsynergy_model.pt"))
model.eval()

# Cell 4: Load and inspect input data
X = pickle.load(open("X.p", "rb"))
print("X shape:", X.shape)

# Cell 5: Run prediction for a single sample
x_sample = torch.tensor(X[0]).float().unsqueeze(0)
with torch.no_grad():
    prediction = model(x_sample)
print("Prediction:", prediction.item())


/var/folders/w3/24s5fhv56rx3q12gkccsw8c80000gn/T/ipykernel_13030/2943307297.py:27: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("deepsynerg

RuntimeError: Error(s) in loading state_dict for DeepSynergyModel:
	size mismatch for model.1.weight: copying a param with shape torch.Size([8192, 8846]) from checkpoint, the shape in current model is torch.Size([8192, 12758]).

In [7]:
# Predict
with torch.no_grad():
    pred = model(x_sample)
print("Prediction:", pred.item())

# Integrated Gradients setup
ig = IntegratedGradients(model)
attributions, delta = ig.attribute(inputs=x_sample, target=None, return_convergence_delta=True)

# Convert to numpy for analysis
attributions = attributions.detach().numpy().flatten()

# Top contributing features
top_indices = np.argsort(-np.abs(attributions))[:20]
top_values = attributions[top_indices]

# Plot top 20 features
plt.figure(figsize=(10, 5))
plt.bar(range(len(top_values)), top_values)
plt.xticks(range(len(top_values)), top_indices, rotation=45)
plt.title("Top 20 Contributing Features (Integrated Gradients)")
plt.xlabel("Feature Index")
plt.ylabel("Attribution Value")
plt.tight_layout()
plt.show()


NameError: name 'x_sample' is not defined

In [9]:
import pickle
X = pickle.load(open("X.p", "rb"))
print("X shape:", X.shape)


X shape: (46104, 12758)
